In [15]:
# Cell 0 — Project Imports

import torch
from torch.nn import functional as F

In [16]:
# Cell 1 — Voxel 좌표와 Physical Distance

def convert_voxel_indices_to_metric_coordinates(
    voxel_indices_zyx: torch.Tensor,  # [N, 3], floating point
    spacing_zyx_mm: tuple[float, float, float],
) -> torch.Tensor:                    # [N, 3], floating point
    """Voxel index를 grid-origin 기준 metric coordinate로 변환."""

    # Python spacing tuple을 입력 좌표와 동일한 device·dtype으로 변환
    spacing_tensor_zyx_mm = torch.tensor(
        spacing_zyx_mm,
        dtype=voxel_indices_zyx.dtype,
        device=voxel_indices_zyx.device,
    )  # [3]

    # 각 Z·Y·X index에 해당 축의 millimeter spacing 적용
    metric_coordinates_zyx_mm = (
        voxel_indices_zyx
        * spacing_tensor_zyx_mm
    )  # [N, 3]

    return metric_coordinates_zyx_mm


def compute_pairwise_physical_distances(
    reference_coordinates_zyx_mm: torch.Tensor,  # [N, 3]
    comparison_coordinates_zyx_mm: torch.Tensor, # [M, 3]
) -> torch.Tensor:                               # [N, M]
    """두 point 집합 사이의 pairwise Euclidean distance 계산."""

    # 모든 reference·comparison point 조합의 물리적 거리 계산
    pairwise_distances_mm = torch.cdist(
        reference_coordinates_zyx_mm,
        comparison_coordinates_zyx_mm,
        p=2.0,
    )  # [N, M]

    return pairwise_distances_mm

# 기준 voxel 한 개 생성
reference_voxel_zyx = torch.tensor(
    [[4.0, 8.0, 8.0]],
    dtype=torch.float32,
)  # [N=1, 3]


# 기준에서 각각 Z축과 X축으로 한 voxel 이동한 좌표 생성
shifted_voxels_zyx = torch.tensor(
    [
        [5.0, 8.0, 8.0],  # Z축 +1 voxel
        [4.0, 8.0, 9.0],  # X축 +1 voxel
    ],
    dtype=torch.float32,
)  # [M=2, 3]


# Z축 간격이 큰 anisotropic spacing 설정
spacing_zyx_mm: tuple[float, float, float] = (
    5.0,
    1.0,
    1.0,
)


# 기준 voxel을 metric coordinate로 변환
reference_coordinate_zyx_mm = (
    convert_voxel_indices_to_metric_coordinates(
        voxel_indices_zyx=reference_voxel_zyx,
        spacing_zyx_mm=spacing_zyx_mm,
    )
)  # [N=1, 3]


# 이동한 voxel들을 metric coordinate로 변환
shifted_coordinates_zyx_mm = (
    convert_voxel_indices_to_metric_coordinates(
        voxel_indices_zyx=shifted_voxels_zyx,
        spacing_zyx_mm=spacing_zyx_mm,
    )
)  # [M=2, 3]


# 기준점과 이동점 사이의 physical distance 계산
pairwise_distances_mm = (
    compute_pairwise_physical_distances(
        reference_coordinates_zyx_mm=(
            reference_coordinate_zyx_mm
        ),
        comparison_coordinates_zyx_mm=(
            shifted_coordinates_zyx_mm
        ),
    )
)  # [N=1, M=2]


print(
    "Reference voxel [N, 3]:       ",
    reference_voxel_zyx,
)

print(
    "Shifted voxels [M, 3]:        ",
    shifted_voxels_zyx,
)

print(
    "Spacing [Z, Y, X] mm:         ",
    spacing_zyx_mm,
)

print(
    "Reference coordinate mm:      ",
    reference_coordinate_zyx_mm,
)

print(
    "Shifted coordinates mm:       ",
    shifted_coordinates_zyx_mm,
)

print(
    "Pairwise distances [N, M] mm: ",
    pairwise_distances_mm,
)

Reference voxel [N, 3]:        tensor([[4., 8., 8.]])
Shifted voxels [M, 3]:         tensor([[5., 8., 8.],
        [4., 8., 9.]])
Spacing [Z, Y, X] mm:          (5.0, 1.0, 1.0)
Reference coordinate mm:       tensor([[20.,  8.,  8.]])
Shifted coordinates mm:        tensor([[25.,  8.,  8.],
        [20.,  8.,  9.]])
Pairwise distances [N, M] mm:  tensor([[5., 1.]])


In [17]:
# Cell 2 — 3D Binary Mask Surface 추출

def extract_binary_surface_6_connected(
    binary_mask: torch.Tensor,  # [B, 1, D, H, W], torch.bool
) -> tuple[
    torch.Tensor,  # Surface mask [B, 1, D, H, W], torch.bool
    torch.Tensor,  # Eroded mask  [B, 1, D, H, W], torch.bool
]:
    """6-connected erosion을 이용한 binary surface 추출."""
    
    # Convolution 연산을 위한 floating-point mask 변환
    binary_mask_float = binary_mask.to(
        dtype=torch.float32,
    )  # [B, 1, D, H, W]
    
    # Center와 여섯 face neighbor를 선택하는 kernel 생성
    erosion_kernel = torch.zeros(
        1,
        1,
        3,
        3,
        3,
        dtype=binary_mask_float.dtype,
        device=binary_mask_float.device,
    )  # [out_channels=1, in_channels=1, 3, 3, 3]
    
    erosion_kernel[0,0,1,1,1] = 1  # center

    erosion_kernel[0,0,0,1,1] = 1  # Z-
    erosion_kernel[0,0,2,1,1] = 1  # Z+

    erosion_kernel[0,0,1,0,1] = 1  # Y-
    erosion_kernel[0,0,1,2,1] = 1  # Y+

    erosion_kernel[0,0,1,1,0] = 1  # X-
    erosion_kernel[0,0,1,1,2] = 1  # X+
    
    # 각 voxel의 center·face-neighbor foreground 개수 계산
    foreground_neighbor_count = F.conv3d(
        input=binary_mask_float,
        weight=erosion_kernel,
        bias=None,
        stride=1,
        padding=1,
    )  # [B, 1, D, H, W]

    # 일곱 위치가 모두 foreground인 interior voxel 선택
    eroded_mask = (
        binary_mask
        & (
            foreground_neighbor_count
            == 7.0
        )
    )  # [B, 1, D, H, W], torch.bool

    # Original foreground에서 eroded interior를 제외해 surface 추출
    surface_mask = (
        binary_mask
        & ~eroded_mask
    )  # [B, 1, D, H, W], torch.bool

    return (
        surface_mask,
        eroded_mask,
    )
    

# 5×5×5 volume 내부에 3×3×3 foreground cube 생성
synthetic_binary_mask = torch.zeros(
    1,
    1,
    5,
    5,
    5,
    dtype=torch.bool,
)  # [B=1, C=1, D=5, H=5, W=5]

synthetic_binary_mask[
    :,
    :,
    1:4,
    1:4,
    1:4,
] = True


# Binary mask에서 surface와 eroded interior 추출
(
    synthetic_surface_mask,
    synthetic_eroded_mask,
) = extract_binary_surface_6_connected(
    binary_mask=synthetic_binary_mask,
)


# Original·interior·surface voxel 수 계산
original_voxel_count = int(
    synthetic_binary_mask.sum().item()
)

interior_voxel_count = int(
    synthetic_eroded_mask.sum().item()
)

surface_voxel_count = int(
    synthetic_surface_mask.sum().item()
)


print(
    "Mask shape:             ",
    synthetic_binary_mask.shape,
)

print(
    "Original voxel count:   ",
    original_voxel_count,
)

print(
    "Eroded interior count:  ",
    interior_voxel_count,
)

print(
    "Surface voxel count:    ",
    surface_voxel_count,
)

print()

# Cube 중앙 Z plane의 original mask 출력
print("Original center plane:")
print(
    synthetic_binary_mask[
        0,
        0,
        2,
    ].to(dtype=torch.int32)
)

print()

# Cube 중앙 Z plane의 eroded interior 출력
print("Eroded center plane:")
print(
    synthetic_eroded_mask[
        0,
        0,
        2,
    ].to(dtype=torch.int32)
)

print()

# Cube 중앙 Z plane의 surface 출력
print("Surface center plane:")
print(
    synthetic_surface_mask[
        0,
        0,
        2,
    ].to(dtype=torch.int32)
)

Mask shape:              torch.Size([1, 1, 5, 5, 5])
Original voxel count:    27
Eroded interior count:   1
Surface voxel count:     26

Original center plane:
tensor([[0, 0, 0, 0, 0],
        [0, 1, 1, 1, 0],
        [0, 1, 1, 1, 0],
        [0, 1, 1, 1, 0],
        [0, 0, 0, 0, 0]], dtype=torch.int32)

Eroded center plane:
tensor([[0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0]], dtype=torch.int32)

Surface center plane:
tensor([[0, 0, 0, 0, 0],
        [0, 1, 1, 1, 0],
        [0, 1, 0, 1, 0],
        [0, 1, 1, 1, 0],
        [0, 0, 0, 0, 0]], dtype=torch.int32)


In [18]:
# Cell 3 — Bidirectional Surface Distance

def extract_single_surface_coordinates_zyx(
    surface_mask: torch.Tensor,  # [B=1, C=1, D, H, W], torch.bool
) -> torch.Tensor:               # [N, 3], torch.float32
    """단일 case·class surface의 ZYX voxel coordinate 추출."""

    # True인 surface voxel의 B·C·Z·Y·X index 추출
    surface_indices_bczyx = torch.nonzero(
        surface_mask,
        as_tuple=False,
    )  # [N, 5]

    # 단일 case·class를 전제로 Z·Y·X column만 선택
    surface_coordinates_zyx = (
        surface_indices_bczyx[
            :,
            2:5,
        ].to(
            dtype=torch.float32,
        )
    )  # [N, 3]

    return surface_coordinates_zyx


def compute_bidirectional_surface_distances(
    target_surface_coordinates_zyx_mm: torch.Tensor,      # [N, 3]
    prediction_surface_coordinates_zyx_mm: torch.Tensor,  # [M, 3]
) -> tuple[
    torch.Tensor,  # Target-to-prediction distances [N]
    torch.Tensor,  # Prediction-to-target distances [M]
]:
    """두 surface 사이의 양방향 nearest distance 계산."""

    # 모든 target·prediction surface point 조합의 거리 계산
    pairwise_distances_mm = (
        compute_pairwise_physical_distances(
            reference_coordinates_zyx_mm=(
                target_surface_coordinates_zyx_mm
            ),
            comparison_coordinates_zyx_mm=(
                prediction_surface_coordinates_zyx_mm
            ),
        )
    )  # [N, M]

    # 각 target point에서 가장 가까운 prediction point 선택
    target_to_prediction_distances_mm = (
        pairwise_distances_mm.min(
            dim=1,
        ).values
    )  # [N]

    # 각 prediction point에서 가장 가까운 target point 선택
    prediction_to_target_distances_mm = (
        pairwise_distances_mm.min(
            dim=0,
        ).values
    )  # [M]

    return (
        target_to_prediction_distances_mm,
        prediction_to_target_distances_mm,
    )
    



# Target cube를 복사해 prediction mask 생성
prediction_binary_mask = (
    synthetic_binary_mask.clone()
)  # [B=1, C=1, D=5, H=5, W=5]

# Target에서 떨어진 corner에 false-positive voxel 추가
prediction_binary_mask[
    0,
    0,
    0,
    0,
    0,
] = True


# Prediction mask의 surface 추출
(
    prediction_surface_mask,
    _,
) = extract_binary_surface_6_connected(
    binary_mask=prediction_binary_mask,
)


# Target surface의 ZYX voxel coordinate 추출
target_surface_coordinates_zyx = (
    extract_single_surface_coordinates_zyx(
        surface_mask=synthetic_surface_mask,
    )
)  # [N=26, 3]


# Prediction surface의 ZYX voxel coordinate 추출
prediction_surface_coordinates_zyx = (
    extract_single_surface_coordinates_zyx(
        surface_mask=prediction_surface_mask,
    )
)  # [M=27, 3]


# 두 surface coordinate를 millimeter 단위로 변환
target_surface_coordinates_zyx_mm = (
    convert_voxel_indices_to_metric_coordinates(
        voxel_indices_zyx=(
            target_surface_coordinates_zyx
        ),
        spacing_zyx_mm=spacing_zyx_mm,
    )
)  # [N=26, 3]

prediction_surface_coordinates_zyx_mm = (
    convert_voxel_indices_to_metric_coordinates(
        voxel_indices_zyx=(
            prediction_surface_coordinates_zyx
        ),
        spacing_zyx_mm=spacing_zyx_mm,
    )
)  # [M=27, 3]


# Target→Prediction과 Prediction→Target 거리 계산
(
    target_to_prediction_distances_mm,
    prediction_to_target_distances_mm,
) = compute_bidirectional_surface_distances(
    target_surface_coordinates_zyx_mm=(
        target_surface_coordinates_zyx_mm
    ),
    prediction_surface_coordinates_zyx_mm=(
        prediction_surface_coordinates_zyx_mm
    ),
)


# 각 방향에서 0보다 큰 distance 개수 계산
target_to_prediction_nonzero_count = int(
    (
        target_to_prediction_distances_mm > 0
    ).sum().item()
)

prediction_to_target_nonzero_count = int(
    (
        prediction_to_target_distances_mm > 0
    ).sum().item()
)


print(
    "Target surface points:              ",
    target_surface_coordinates_zyx.shape[0],
)

print(
    "Prediction surface points:          ",
    prediction_surface_coordinates_zyx.shape[0],
)

print(
    "Target → prediction non-zero count: ",
    target_to_prediction_nonzero_count,
)

print(
    "Prediction → target non-zero count: ",
    prediction_to_target_nonzero_count,
)

print(
    "Target → prediction maximum mm:     ",
    target_to_prediction_distances_mm.max().item(),
)

print(
    "Prediction → target maximum mm:     ",
    prediction_to_target_distances_mm.max().item(),
)

Target surface points:               26
Prediction surface points:           27
Target → prediction non-zero count:  0
Prediction → target non-zero count:  1
Target → prediction maximum mm:      0.0
Prediction → target maximum mm:      5.196152210235596


In [19]:
# Cell 4 — Normalized Surface Dice

def compute_point_based_normalized_surface_dice(
    target_to_prediction_distances_mm: torch.Tensor,  # [N]
    prediction_to_target_distances_mm: torch.Tensor,  # [M]
    tolerance_mm: float,
) -> torch.Tensor:                                    # [], torch.float32
    """양방향 surface point의 tolerance 통과 비율 계산."""

    # 음수 tolerance 차단
    if tolerance_mm < 0:
        raise ValueError(
            "tolerance_mm는 0 이상이어야 합니다."
        )

    # Empty surface의 평가 정책을 이번 Cell에서 임의 결정하지 않도록 차단
    if (
        target_to_prediction_distances_mm.numel() == 0
        or prediction_to_target_distances_mm.numel() == 0
    ):
        raise ValueError(
            "Empty surface 처리는 Part 5.3의 명시적 정책이 필요합니다."
        )

    # Target surface 중 prediction에서 tolerance 이내인 point 선택
    target_surface_within_tolerance = (
        target_to_prediction_distances_mm
        <= tolerance_mm
    )  # [N], torch.bool

    # Prediction surface 중 target에서 tolerance 이내인 point 선택
    prediction_surface_within_tolerance = (
        prediction_to_target_distances_mm
        <= tolerance_mm
    )  # [M], torch.bool

    # 양방향에서 tolerance를 통과한 surface point 수 계산
    matched_surface_point_count = (
        target_surface_within_tolerance.sum()
        + prediction_surface_within_tolerance.sum()
    )  # [], torch.long

    # 양방향 전체 surface point 수 계산
    total_surface_point_count: int = (
        target_to_prediction_distances_mm.numel()
        + prediction_to_target_distances_mm.numel()
    )

    # 통과 point 수를 전체 point 수로 정규화
    normalized_surface_dice = (
        matched_surface_point_count.to(
            dtype=torch.float32,
        )
        / total_surface_point_count
    )  # []

    return normalized_surface_dice


# Extra false-positive distance의 경계 전후 tolerance 설정
tolerances_mm: tuple[float, ...] = (
    0.0,
    5.0,
    6.0,
)


# 각 tolerance에서 point-based NSD 계산
for tolerance_mm in tolerances_mm:
    normalized_surface_dice = (
        compute_point_based_normalized_surface_dice(
            target_to_prediction_distances_mm=(
                target_to_prediction_distances_mm
            ),
            prediction_to_target_distances_mm=(
                prediction_to_target_distances_mm
            ),
            tolerance_mm=tolerance_mm,
        )
    )  # []

    print(
        f"Tolerance = {tolerance_mm:3.1f} mm | "
        f"NSD = {normalized_surface_dice.item():.6f}"
    )

Tolerance = 0.0 mm | NSD = 0.981132
Tolerance = 5.0 mm | NSD = 0.981132
Tolerance = 6.0 mm | NSD = 1.000000


In [20]:
# Cell 5 — Maximum Hausdorff Distance와 HD95

def compute_maximum_hausdorff_and_hd95(
    target_to_prediction_distances_mm: torch.Tensor,  # [N]
    prediction_to_target_distances_mm: torch.Tensor,  # [M]
) -> tuple[
    torch.Tensor,  # Combined bidirectional distances [N+M]
    torch.Tensor,  # Maximum Hausdorff distance []
    torch.Tensor,  # HD95 []
]:
    """양방향 surface distance의 maximum과 95th percentile 계산."""

    # Empty surface policy를 이번 Cell에서 임의 결정하지 않도록 차단
    if (
        target_to_prediction_distances_mm.numel() == 0
        or prediction_to_target_distances_mm.numel() == 0
    ):
        raise ValueError(
            "Empty surface 처리는 Part 5.3의 명시적 정책이 필요합니다."
        )

    # 두 방향의 nearest distance를 하나의 vector로 결합
    combined_surface_distances_mm = torch.cat(
        (
            target_to_prediction_distances_mm,
            prediction_to_target_distances_mm,
        ),
        dim=0,
    )  # [N+M]

    # 가장 큰 단일 surface error 선택
    maximum_hausdorff_distance_mm = (
        combined_surface_distances_mm.max()
    )  # []

    # 결합된 양방향 distance의 95th percentile 계산
    hd95_mm = torch.quantile(
        combined_surface_distances_mm,
        q=0.95,
    )  # []

    return (
        combined_surface_distances_mm,
        maximum_hausdorff_distance_mm,
        hd95_mm,
    )


# 현재 extra false-positive example의 Hausdorff metric 계산
(
    combined_surface_distances_mm,
    maximum_hausdorff_distance_mm,
    hd95_mm,
) = compute_maximum_hausdorff_and_hd95(
    target_to_prediction_distances_mm=(
        target_to_prediction_distances_mm
    ),
    prediction_to_target_distances_mm=(
        prediction_to_target_distances_mm
    ),
)


# 양방향 합계 100개 중 6개가 10 mm인 지속적 오류 생성
persistent_target_to_prediction_distances_mm = torch.cat(
    (
        torch.zeros(
            47,
            dtype=torch.float32,
        ),
        torch.full(
            (3,),
            10.0,
            dtype=torch.float32,
        ),
    ),
    dim=0,
)  # [N=50]

persistent_prediction_to_target_distances_mm = torch.cat(
    (
        torch.zeros(
            47,
            dtype=torch.float32,
        ),
        torch.full(
            (3,),
            10.0,
            dtype=torch.float32,
        ),
    ),
    dim=0,
)  # [M=50]


# Surface의 6%에서 발생한 persistent error 평가
(
    persistent_combined_distances_mm,
    persistent_maximum_hausdorff_mm,
    persistent_hd95_mm,
) = compute_maximum_hausdorff_and_hd95(
    target_to_prediction_distances_mm=(
        persistent_target_to_prediction_distances_mm
    ),
    prediction_to_target_distances_mm=(
        persistent_prediction_to_target_distances_mm
    ),
)


print(
    "Current distance count:       ",
    combined_surface_distances_mm.numel(),
)

print(
    "Current non-zero count:       ",
    (
        combined_surface_distances_mm > 0
    ).sum().item(),
)

print(
    "Current maximum HD mm:        ",
    maximum_hausdorff_distance_mm.item(),
)

print(
    "Current HD95 mm:              ",
    hd95_mm.item(),
)

print()

print(
    "Persistent distance count:    ",
    persistent_combined_distances_mm.numel(),
)

print(
    "Persistent non-zero count:    ",
    (
        persistent_combined_distances_mm > 0
    ).sum().item(),
)

print(
    "Persistent maximum HD mm:     ",
    persistent_maximum_hausdorff_mm.item(),
)

print(
    "Persistent HD95 mm:           ",
    persistent_hd95_mm.item(),
)

Current distance count:        53
Current non-zero count:        1
Current maximum HD mm:         5.196152210235596
Current HD95 mm:               0.0

Persistent distance count:     100
Persistent non-zero count:     6
Persistent maximum HD mm:      10.0
Persistent HD95 mm:            10.0
